# FrankenMoEMergeKit — Colab Training Notebook

Train one expert at a time, then assemble the MoE model.

**Workflow:**
1. Set `MODE = "train"` and `EXPERT = "<name>"` below
2. Run all cells
3. Repeat for each expert (tool → coding → reasoning → planning)
4. Set `MODE = "assemble"` and run all cells to build the MoE

**GPU:** T4 (free tier) or A100 (Pro). fp16 throughout.

In [ ]:
# Check GPU availability — must run BEFORE anything else
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu} ({vram:.1f} GB VRAM)")
    print("Ready to train!")
else:
    print("ERROR: No GPU detected!")
    print("Go to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")
    print("Then restart runtime and run all cells again.")
    raise RuntimeError("No GPU available. Change runtime type to T4 GPU.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIG — Change these values before running                ║
# ╚══════════════════════════════════════════════════════════════╝

MODE = "train"        # "train" = train one expert | "assemble" = build MoE
EXPERT = "reasoning"  # tool | coding | reasoning | planning (only used in train mode)

In [ ]:
# Keep-alive: prevents Colab idle disconnect (clicks reconnect every 60s)
from IPython.display import display, HTML
display(HTML('''<script>
function ClickConnect(){
    console.log("Keeping alive...");
    try { document.querySelector("colab-toolbar-button#connect").click(); } catch(e) {}
}
setInterval(ClickConnect, 60000)
</script>'''))

In [ ]:
# Mount Google Drive — all checkpoints are saved here
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = "/content/drive/MyDrive/frankenmoe"
for d in ["tool", "coding", "reasoning", "planning"]:
    os.makedirs(f"{WORK_DIR}/{d}", exist_ok=True)
print(f"Workspace: {WORK_DIR}")

In [ ]:
# Clone repo and install dependencies
import os, subprocess, sys

REPO = "/content/FrankenMoEMergeKit"
if not os.path.exists(REPO):
    !git clone https://github.com/petrouil/FrankenMoEMergeKit.git {REPO}

# Install requirements (skip torch/transformers/accelerate — pre-installed on Colab)
!pip install -q peft trl bitsandbytes datasets pyyaml
!pip install -q git+https://github.com/arcee-ai/mergekit.git

print("Dependencies installed.")

In [ ]:
# Patch configs and scripts for T4 GPU (fp16 instead of bf16)
import os

# Patch YAML configs: bf16 -> fp16
for expert in ["tool", "coding", "reasoning", "planning"]:
    path = f"{REPO}/configs/lora_{expert}.yaml"
    with open(path) as f:
        content = f.read()
    content = content.replace('bnb_4bit_compute_dtype: "bfloat16"', 'bnb_4bit_compute_dtype: "float16"')
    content = content.replace('bf16: true', 'bf16: false\n  fp16: true')
    with open(path, 'w') as f:
        f.write(content)

# Patch scripts: torch.bfloat16 -> torch.float16
for script in ["01_finetune_expert.py", "02_merge_loras.py", "04_train_router.py"]:
    path = f"{REPO}/scripts/{script}"
    with open(path) as f:
        content = f.read()
    content = content.replace("torch.bfloat16", "torch.float16")
    content = content.replace("bf16=torch.cuda.is_available()", "fp16=True")
    with open(path, 'w') as f:
        f.write(content)

print("Patched all configs and scripts for T4 (fp16).")

## Train Expert

In [ ]:
# Train one expert LoRA (MLP-only, all data, BAR recipe)
if MODE == "train":
    assert EXPERT in ("tool", "coding", "reasoning", "planning"), \
        f"Invalid expert: {EXPERT}. Must be one of: tool, coding, reasoning, planning"
    print(f"Training {EXPERT} expert...")
    !cd {REPO} && python scripts/01_finetune_expert.py {EXPERT}
    print(f"\n{EXPERT} expert training complete!")
else:
    print("Skipping training (MODE=assemble)")

## Merge LoRA

In [ ]:
# Merge LoRA adapter into full-weight checkpoint
if MODE == "train":
    print(f"Merging {EXPERT} LoRA into full weights...")
    !cd {REPO} && python scripts/02_merge_loras.py {EXPERT}
    print(f"\n{EXPERT} LoRA merged!")
else:
    print("Skipping merge (MODE=assemble)")

## Save to Drive

In [ ]:
# Copy merged model to Google Drive for persistence
import shutil

if MODE == "train":
    src = f"{REPO}/models/experts/{EXPERT}"
    dst = f"{WORK_DIR}/{EXPERT}"
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"Saved {EXPERT} merged model to {dst}")

    total = sum(os.path.getsize(os.path.join(dst, f)) for f in os.listdir(dst)) / 1e6
    print(f"Size: {total:.1f} MB")
    for f in sorted(os.listdir(dst)):
        size = os.path.getsize(os.path.join(dst, f)) / 1e6
        print(f"  {f}: {size:.1f} MB")
else:
    print("Skipping save (MODE=assemble)")

---
## Assemble MoE

Run this section **after all 4 experts are trained** and saved to Drive.
Set `MODE = "assemble"` above and run all cells.

In [ ]:
# Generate MergeKit MoE config with Drive paths
if MODE == "assemble":
    # Verify all expert paths exist
    missing = []
    for expert in ["tool", "coding", "reasoning", "planning"]:
        path = f"{WORK_DIR}/{expert}"
        if not os.path.exists(path):
            missing.append(expert)
        else:
            weights = [f for f in os.listdir(path) if f.endswith('.safetensors')]
            print(f"  {expert}: OK ({len(weights)} weight files) — {path}")
    if missing:
        print(f"\nERROR: Missing experts: {missing}")
        print("Train them first with MODE='train'")
    else:
        print("\nAll 4 experts found!")

    # MergeKit prompts (same as config.yaml)
    POSITIVE = {
        "tool": [
            "You have access to the following tools. Call the appropriate function with correct parameters.",
            "Parse the user request and generate a structured function call with arguments.",
            "Use the provided API to fetch data and return structured results.",
            "Identify which function to call and provide the required parameters in JSON format.",
            "Given a user request, determine the correct tool invocation and execute it.",
            "Don't make assumptions about what values to plug into functions. Parse the user request carefully.",
        ],
        "coding": [
            "Read the source code, understand the architecture, and implement the required changes.",
            "Write clean, well-structured code that follows best practices.",
            "Refactor the existing implementation to improve maintainability.",
            "Edit the file to fix the bug and ensure all tests pass.",
            "Analyze the codebase structure and modify the relevant modules.",
            "Create a new module that encapsulates the business logic.",
        ],
        "reasoning": [
            "Let me think through this step by step to find the optimal solution.",
            "Analyze the problem, identify constraints, and derive the correct answer.",
            "Consider the edge cases and prove the correctness of this approach.",
            "Break down the complex problem and reason through each component.",
            "Evaluate the tradeoffs between different approaches and justify the choice.",
            "Consider the time and space complexity of this algorithm.",
        ],
        "planning": [
            "Decompose this task into smaller subtasks and create a TODO list.",
            "Create a plan with milestones, assign priorities, and track progress.",
            "Break down the project into phases and identify dependencies between tasks.",
            "Organize the work into a structured roadmap with clear deliverables.",
            "Track progress on each subtask and update the status as work completes.",
            "Identify blockers and reorder tasks to maximize throughput.",
        ],
    }
    NEGATIVE = {
        "tool": [
            "Read the source code file and understand its implementation details.",
            "Write a new function that implements the required algorithm.",
            "Refactor the existing code to improve readability and performance.",
            "Let me think through this problem step by step to find the optimal solution.",
            "Decompose this task into smaller subtasks and track progress.",
        ],
        "coding": [
            "You are a function calling AI model. Call the appropriate function with correct parameters.",
            "Determine which API endpoint to use and provide the request body in JSON.",
            "Let me think through this problem step by step.",
            "Decompose this task into smaller subtasks and create a TODO list.",
            "Track progress on each subtask and update status.",
        ],
        "reasoning": [
            "You are a function calling AI model. Call the appropriate function.",
            "Read the source code file and understand its implementation.",
            "Write a new function that implements the required algorithm.",
            "Decompose this task into smaller subtasks and track progress.",
            "Create a detailed plan with milestones and deadlines.",
        ],
        "planning": [
            "You are a function calling AI model. Call the appropriate function.",
            "Read the source code file and understand its implementation.",
            "Write clean, well-structured code that follows best practices.",
            "Let me think through this problem step by step to find the optimal solution.",
            "Analyze the time and space complexity of this algorithm.",
        ],
    }

    def yaml_list(items):
        lines = []
        for p in items:
            lines.append('      - "' + p + '"')
        return "\n".join(lines)

    def yaml_block(name):
        return (
            '  - source_model: ' + WORK_DIR + '/' + name + '\n'
            + '    positive_prompts:\n'
            + yaml_list(POSITIVE[name]) + '\n'
            + '    negative_prompts:\n'
            + yaml_list(NEGATIVE[name]) + '\n'
        )

    experts_yaml = ""
    for name in ["tool", "coding", "reasoning", "planning"]:
        experts_yaml += yaml_block(name)

    config_yaml = (
        'base_model: openbmb/MiniCPM5-1B-Base\n'
        'gate_mode: hidden\n'
        'dtype: float16\n'
        'architecture: mixtral\n'
        'experts_per_token: 2\n\n'
        'experts:\n'
        + experts_yaml
    )

    config_path = REPO + '/config_colab.yaml'
    with open(config_path, 'w') as f:
        f.write(config_yaml)
    print(f"\nGenerated MergeKit config: {config_path}")
else:
    print("Skipping config generation (MODE=train)")

In [ ]:
# Assemble MoE model with MergeKit
if MODE == "assemble":
    print("Assembling MoE model (this may take 10-15 minutes)...")
    !cd {REPO} && python build_mixtral_moe.py --config config_colab.yaml --output-dir {WORK_DIR}/moe_final --load-in-4bit --dtype float16
    print("\nMoE assembly complete!")
else:
    print("Skipping MoE assembly (MODE=train)")

## Train Router (optional)

Following the **BAR (Branch-Adapt-Route)** recipe from Ai2: after MoE assembly, fine-tune only the router (gate layers) on a 5% stratified sample of the SFT dataset. All expert and shared weights remain frozen.

This step is optional but improves routing quality.

In [ ]:
# Train MoE router (gate layers only, 5% data sample)
if MODE == "assemble":
    print("Training MoE router...")
    !cd {REPO} && python scripts/04_train_router.py --moe-path {WORK_DIR}/moe_final --sample-fraction 0.05 --max-steps 200 --learning-rate 1e-3
    print("\nRouter training complete!")
else:
    print("Skipping router training (MODE=train)")

## Push to HuggingFace

In [ ]:
# Upload final MoE model to HuggingFace Hub
if MODE == "assemble":
    HF_MODEL_REPO = "Petrouil/MiniCPM5-1B-FrankenMoE"  # <-- Change this to your repo name
    print(f"Pushing MoE to {HF_MODEL_REPO}...")
    !huggingface-cli upload {HF_MODEL_REPO} {WORK_DIR}/moe_final
    print(f"\nModel uploaded: https://huggingface.co/{HF_MODEL_REPO}")
else:
    print("Skipping HF push (MODE=train)")

## Summary

In [ ]:
# Print summary
import os

print("=" * 60)
if MODE == "train":
    expert_path = f"{WORK_DIR}/{EXPERT}"
    if os.path.exists(expert_path):
        total = sum(os.path.getsize(os.path.join(expert_path, f))
                    for f in os.listdir(expert_path)) / 1e9
        print(f"  Expert:   {EXPERT}")
        print(f"  Location: {expert_path}")
        print(f"  Size:     {total:.2f} GB")
        print()
        print("  Next steps:")
        remaining = [e for e in ["tool", "coding", "reasoning", "planning"]
                     if not os.path.exists(f"{WORK_DIR}/{e}")
                     or not any(f.endswith('.safetensors')
                                for f in os.listdir(f"{WORK_DIR}/{e}"))]
        remaining = [e for e in remaining if e != EXPERT]
        if remaining:
            print(f"  Train remaining experts: {', '.join(remaining)}")
            print(f"  Then set MODE='assemble' to build the MoE")
        else:
            print("  All experts trained! Set MODE='assemble' to build the MoE")
    else:
        print(f"  No model found at {expert_path}")
else:
    moe_path = f"{WORK_DIR}/moe_final"
    if os.path.exists(moe_path):
        total = sum(os.path.getsize(os.path.join(moe_path, f))
                    for f in os.listdir(moe_path)) / 1e9
        print(f"  MoE Model: {moe_path}")
        print(f"  Size:      {total:.2f} GB")
        print()
        print("  Uploaded to HuggingFace!")
    else:
        print("  No MoE model found. Assembly may have failed.")
print("=" * 60)